# Pandas Basics

## Overview

Pandas is the foundation of data wrangling in Python. Two core data structures:

| Structure | Description | R equivalent |
|---|---|---|
| `Series` | 1D labelled array | Named vector |
| `DataFrame` | 2D labelled table | `data.frame` / tibble |

**Core workflow:** load → inspect → select → filter → mutate → summarise → export.

**Two indexing systems:**
- `.loc[]` — label-based; uses row/column names
- `.iloc[]` — position-based; uses integer positions

Always be explicit about which you are using. Mixing them is a common source of silent errors.

---

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.3f}'.format)

rng = np.random.default_rng(42)

# Simulate: 120 riparian monitoring sites
n = 120
sites = pd.DataFrame({
    'site_id':    [f'SITE_{i:03d}' for i in range(1, n+1)],
    'catchment':  rng.choice(['North', 'South', 'East', 'West'], size=n),
    'elevation':  rng.uniform(50, 400, n).round(1),
    'nitrate':    rng.gamma(2, 2, n).round(2),
    'phosphorus': rng.gamma(1.5, 1.5, n).round(2),
    'richness':   rng.integers(5, 35, n),
    'treatment':  rng.choice(['control', 'restored'], size=n),
})

print(f'Shape: {sites.shape}')
sites.head()

---
## Inspecting a DataFrame

In [ ]:
sites.info()                                          # dtypes, non-null counts
print(sites.describe())                               # numeric summary
print(sites['catchment'].value_counts())              # categorical counts
print(sites['treatment'].value_counts(normalize=True).round(3))  # proportions
print('\nMissing values:')
print(sites.isnull().sum())

---
## Selecting Rows and Columns

In [ ]:
# Column selection
sites[['site_id', 'richness', 'treatment']].head(3)

# Select columns by dtype
numeric_cols = sites.select_dtypes(include='number').columns.tolist()
print('Numeric columns:', numeric_cols)

# Boolean filtering with .loc
high_nitrate = sites.loc[sites['nitrate'] > 5]
print(f'High nitrate sites: {len(high_nitrate)}')

# Multiple conditions — each wrapped in parentheses
restored_north = sites.loc[
    (sites['treatment'] == 'restored') & (sites['catchment'] == 'North')
]
print(f'Restored North sites: {len(restored_north)}')

# .iloc for positional access
print('\nFirst 3 rows, first 4 columns:')
print(sites.iloc[:3, :4])

# query() for readable filtering
rich_restored = sites.query('richness > 25 and treatment == "restored"')
print(f'Rich restored sites: {len(rich_restored)}')

---
## Creating and Transforming Columns

In [ ]:
# assign(): method-chaining friendly — equivalent to dplyr::mutate
sites = (
    sites
    .assign(
        log_nitrate    = lambda df: np.log1p(df['nitrate']),
        nutrient_ratio = lambda df: df['nitrate'] / (df['phosphorus'] + 0.01),
        elev_class     = lambda df: pd.cut(
            df['elevation'],
            bins=[0, 150, 300, np.inf],
            labels=['low', 'mid', 'high']
        ),
        richness_std   = lambda df: (
            (df['richness'] - df['richness'].mean()) / df['richness'].std()
        )
    )
)
print(sites[['nitrate', 'log_nitrate', 'nutrient_ratio', 'elev_class']].head(5))

# np.where() for conditional column — faster than apply() for binary logic
sites['richness_label'] = np.where(
    sites['richness'] > 25, 'high',
    np.where(sites['richness'] < 12, 'low', 'medium')
)
print(sites['richness_label'].value_counts())

---
## Grouping and Summarising

In [ ]:
# groupby + agg: named aggregation
summary = (
    sites
    .groupby(['catchment', 'treatment'])
    .agg(
        n             = ('site_id',   'count'),
        richness_mean = ('richness',  'mean'),
        richness_sd   = ('richness',  'std'),
        nitrate_median= ('nitrate',   'median'),
    )
    .round(2)
    .reset_index()   # promotes grouping columns back to regular columns
)
print(summary)

# transform(): adds group summary back to original row count
# Equivalent to dplyr::mutate after group_by
sites['catchment_mean_richness'] = (
    sites.groupby('catchment')['richness'].transform('mean')
)
sites['richness_within_group'] = (
    sites['richness'] - sites['catchment_mean_richness']
)
print(sites[['site_id','catchment','richness','catchment_mean_richness','richness_within_group']].head(5))

---
## Sorting and Ranking

In [ ]:
sites_sorted = sites.sort_values(
    ['catchment', 'richness'], ascending=[True, False]
)

# Rank within groups
sites['richness_rank'] = (
    sites.groupby('catchment')['richness']
    .rank(method='dense', ascending=False)
    .astype(int)
)

print(f'Final shape: {sites.shape}')
print(f'Columns: {sites.columns.tolist()}')

---

## Common Pitfalls

**1. Chained indexing (`df['col'][mask]`) instead of `.loc`**  
Chained indexing may operate on a copy rather than the original DataFrame — assignments silently fail and pandas raises a `SettingWithCopyWarning`. Always use `df.loc[condition, 'col']` for combined row filtering and column assignment.

**2. Modifying a DataFrame during iteration**  
Iterating with `iterrows()` and modifying the DataFrame in the loop is slow and error-prone. Almost all row-wise operations have a vectorised pandas or numpy equivalent. If custom logic is unavoidable, use `assign()` or `apply()` rather than modifying in place during iteration.

**3. Forgetting `reset_index()` after `groupby().agg()`**  
After aggregation, grouping columns become the index. Accessing them as regular columns then raises a `KeyError`. Always call `.reset_index()` after aggregation unless you specifically need a MultiIndex.

**4. Using `apply()` where vectorised operations exist**  
`df['col'].apply(lambda x: x * 2)` is orders of magnitude slower than `df['col'] * 2`. `apply()` is a Python loop under the hood. Reserve it for genuinely complex per-row logic that cannot be expressed vectorially; prefer `np.where()` for conditionals and built-in pandas methods for everything else.

**5. Not checking dtypes after loading**  
Pandas infers dtypes on read. Numeric columns stored as strings, integers read as floats due to missing values, and categories read as `object` are all common. Always run `.info()` immediately after loading and cast explicitly with `.astype()` before proceeding.

---
*python_methods_library · Samantha McGarrigle · [github.com/samantha-mcgarrigle](https://github.com/samantha-mcgarrigle)*